In [1]:
!pip install peft

In [2]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import load_dataset, Dataset
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
import torch

In [3]:
!unzip mbart_lora_es_pt.zip -d mbart_lora_es_pt

Archive:  mbart_lora_es_pt.zip
replace mbart_lora_es_pt/mbart_lora_es_pt/adapter_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/adapter_config.json  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/adapter_model.safetensors  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/checkpoint-3375/adapter_config.json  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/checkpoint-3375/adapter_model.safetensors  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/checkpoint-3375/optimizer.pt  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/checkpoint-3375/README.md  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/checkpoint-3375/rng_state.pth  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/checkpoint-3375/scaler.pt  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/checkpoint-3375/scheduler.pt  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/checkpoint-3375/sentencepiece.bpe.model  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/checkpoint-3375/special_t

In [4]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
base_model = MBartForConditionalGeneration.from_pretrained(model_name)
model = PeftModel.from_pretrained(base_model, "./mbart_lora_es_pt/mbart_lora_es_pt")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
model.train()
for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True

model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 612,059,136 || trainable%: 0.1927


In [6]:
src_lang = "es_XX"
tgt_lang = "pt_XX"
tokenizer.src_lang = src_lang

# Función de traducción
def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Indicamos que la secuencia de salida debe comenzar en portugués
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
        max_length=128
    )

    traduccion = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    return traduccion

In [7]:
frases = [
    "Hola mundo",
    "¿Cómo estás?",
    "Me gusta aprender cosas nuevas.",
    "Estoy entrenando un modelo de traducción.",
    "La inteligencia artificial es fascinante."
]

for f in frases:
    print(f"ES: {f}")
    print(f"PT: {traducir(f)}\n")

ES: Hola mundo
PT: O mundo ainda não o sente!

ES: ¿Cómo estás?
PT: Como é que estás?

ES: Me gusta aprender cosas nuevas.
PT: Gosto de aprender coisas novas.

ES: Estoy entrenando un modelo de traducción.
PT: Estou a praticar um modelo de tradução.

ES: La inteligencia artificial es fascinante.
PT: A inteligência artificial é fascinante.



In [8]:
!pip install -U datasets

In [9]:
import huggingface_hub
huggingface_hub.login() # now you will be prompted to enter your token; enter it.

In [11]:
ds_es = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="dev")
ds_lit = load_dataset("openlanguagedata/flores_plus", "azj_Latn", split="dev")
parallel_lit = [{"translation": {"es": e["text"], "azj": g["text"]}} for e, g in zip(ds_es, ds_lit)]

dataset_lit = Dataset.from_list(parallel_lit).train_test_split(test_size=0.1, seed=42)
train_lit = dataset_lit["train"]
eval_lit = dataset_lit["test"]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

spa_Latn.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

spa_Latn.parquet:   0%|          | 0.00/134k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

...

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

azj_Latn.parquet:   0%|          | 0.00/128k [00:00<?, ?B/s]

azj_Latn.parquet:   0%|          | 0.00/134k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [13]:
tokenizer.src_lang = "es_XX"
tokenizer.tgt_lang = "az_AZ"

def tokenize_gl(batch):
    src = [x["es"] for x in batch["translation"]]
    tgt = [x["azj"] for x in batch["translation"]]
    inputs = tokenizer(src, truncation=True, padding="max_length", max_length=128)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(tgt, truncation=True, padding="max_length", max_length=128)
    inputs["labels"] = labels["input_ids"]
    return inputs

train_tokenized_az = train_lit.map(tokenize_gl, batched=True)
eval_tokenized_az = eval_lit.map(tokenize_gl, batched=True)

Map:   0%|          | 0/897 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [14]:
training_args_az = Seq2SeqTrainingArguments(
    output_dir="./mbart_lora_es_az",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-4,
    num_train_epochs=15,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs_az",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none",
    label_names=["labels"]
)

trainer_az = Seq2SeqTrainer(
    model=model,
    args=training_args_az,
    train_dataset=train_tokenized_az,
    eval_dataset=eval_tokenized_az,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

In [15]:
trainer_az.train()
model.save_pretrained("./mbart_lora_es_az")
tokenizer.save_pretrained("./mbart_lora_es_az")

Epoch,Training Loss,Validation Loss
1,No log,8.675253
2,No log,8.664565
3,8.633400,8.663382
4,8.633400,8.671215
5,8.463800,8.677733
6,8.463800,8.699136
7,8.348300,8.697938
8,8.348300,8.710940
9,8.263800,8.711054
10,8.263800,8.732935


('./mbart_lora_es_az/tokenizer_config.json',
 './mbart_lora_es_az/special_tokens_map.json',
 './mbart_lora_es_az/sentencepiece.bpe.model',
 './mbart_lora_es_az/added_tokens.json',
 './mbart_lora_es_az/tokenizer.json')

In [16]:
pip install evaluate sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.3 MB/s eta 0:00:00


In [17]:
from evaluate import load

In [18]:
def test_translation_batch(sentences_es, references_gl, model_path="./mbart_lora_es_az"):
    try:
        # Cargar modelo base + adaptadores LoRA
        base_model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
        model = PeftModel.from_pretrained(base_model, model_path)
        model.eval()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)

        # Cargar tokenizer
        tokenizer = MBart50TokenizerFast.from_pretrained(model_path)
        inputs = tokenizer(sentences_es, return_tensors="pt", padding=True, truncation=True).to(device)

        # Generar traducciones
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=128,
                num_beams=4,
                early_stopping=True,
                do_sample=False
            )

        translations = [tokenizer.decode(out, skip_special_tokens=True) for out in outputs]

        # Mostrar resultados
        for src, pred, ref in zip(sentences_es, translations, references_gl):
            print(f"ES: {src}")
            print(f"AZ (pred): {pred}")
            print(f"AZ (ref):  {ref}")
            print("-" * 60)

        # Calcular métricas BLEU y chrF en lote
        print("\n📊 Métricas globales:")
        bleu = load("bleu")
        chrf = load("chrf")

        bleu_score = bleu.compute(predictions=translations, references=[[r] for r in references_gl])
        chrf_score = chrf.compute(predictions=translations, references=references_gl)

        print(f"BLEU: {bleu_score['bleu']:.4f}")
        print(f"chrF: {chrf_score['score']:.2f}")

    except Exception as e:
        print(f"❌ Error en traducción: {e}")

In [19]:
ds_es = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="devtest[:50]")
ds_azj = load_dataset("openlanguagedata/flores_plus", "azj_Latn", split="devtest[:50]")

# Tomar una muestra de 50 ejemplos para visualización
sample_es = ds_es["text"][:50]
sample_azj = ds_azj["text"][:50]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

In [20]:
test_translation_batch(sample_es, sample_azj)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


ES: «Actualmente, tenemos ratones de cuatro meses de edad que antes solían ser diabéticos y que ya no lo son», agregó.
AZ (pred): "Şu anda, əvvəllər diabetik xəstəlikdə idik və indi daha diabetik olmayan 4 aylıq əkrarlarımız var", dedi.
AZ (ref):  "Hazırda bizim diabetdən əziyyət çəkməyən, amma əvvəl diabet olan 4 aylıq siçanlarımız var", deyə o əlavə etdi.
------------------------------------------------------------
ES: La investigación todavía se ubica en su etapa inicial, conforme indicara el Dr. Ehud Ur, docente en la carrera de medicina de la Universidad de Dalhousie, en Halifax, Nueva Escocia, y director del departamento clínico y científico de la Asociación Canadiense de Diabetes.
AZ (pred): Dalusini-Halifax, Yeni-Escociadakı Dalhuzi universitetinin tıbbi kariyeri öyrətən və Kanada Diabetik Cəmiyyətinin tıbbi və elmi üzrə dairə Direktoru Dr. Ehud Ur qeyd edir ki, araşdırmalar hələ başlangıç safhasındadır.
AZ (ref):  Yeni Şotlandiya əyalətinin paytaxtı Halifaksdaki Delxauzi unive

BLEU: 0.0598
chrF: 33.41


In [23]:
import shutil
from google.colab import files

# Comprimir la carpeta (reemplaza 'nombre_de_tu_carpeta')
shutil.make_archive('mbart_lora_es_az', 'zip', 'mbart_lora_es_az')

# Descargar el archivo ZIP
files.download('mbart_lora_es_az.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>